# CP2K band unfolding

This viewer loads precomputed unfolding weights retrieved by the CP2K SCF workchain.


In [ ]:
%load_ext aiida
%aiida

import urllib.parse as urlparse

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from aiida import orm


In [ ]:
def _query_pk(default=None):
    try:
        query = urlparse.parse_qs(urlparse.urlparse(jupyter_notebook_url).query)
        if "pk" in query:
            return int(query["pk"][0])
    except Exception:
        pass
    return default

pk = _query_pk()
if pk is None:
    raise ValueError("Open this viewer from a CP2K SCF unfolding result, or pass ?pk=<workchain_pk>.")

workcalc = orm.load_node(pk)
if "unfolding_retrieved" not in workcalc.outputs:
    raise ValueError("This SCF workchain has no unfolding_retrieved output. Submit with Compute band unfolding enabled.")

with workcalc.outputs.unfolding_retrieved.open("unfolding_bands.npz", "rb") as handle:
    loaded = np.load(handle)
    data = {key: loaded[key] for key in loaded.files}

projection_data = None
projection_filename = str(data.get("pdos_projection_filename", "unfolding_projections.npz"))
retrieved_names = workcalc.outputs.unfolding_retrieved.base.repository.list_object_names()
if projection_filename in retrieved_names:
    with workcalc.outputs.unfolding_retrieved.open(projection_filename, "rb") as handle:
        loaded = np.load(handle)
        projection_data = {key: loaded[key] for key in loaded.files}


def projection_weights_by_mo(spin, atom_indices=None, channels=None):
    if projection_data is None:
        return None
    mask = projection_data["spin"] == spin
    if atom_indices is not None:
        atom_indices = np.asarray(atom_indices, dtype=int)
        mask &= np.isin(projection_data["atom_index"], atom_indices)
    if channels is not None:
        channel_names = projection_data["channels"].astype(str)
        selected = set(channels)
        channel_mask = np.asarray([
            name in selected or name[:1] in selected
            for name in channel_names
        ])
        mask &= channel_mask[projection_data["channel_index"]]
    weights = np.zeros_like(data[f"evals_ev_spin_{spin}"], dtype=float)
    mask &= projection_data["mo_index"] < len(weights)
    np.add.at(weights, projection_data["mo_index"][mask], projection_data["projection"][mask])
    return weights

def _parse_atom_indices(text):
    text = str(text).strip().replace(",", " ").replace(";", " ")
    if not text:
        return None
    indices = []
    for item in text.split():
        item = item.replace(" ", "")
        if ".." in item:
            start, end = [int(x) for x in item.split("..", 1)]
            if end < start:
                start, end = end, start
            indices.extend(range(start - 1, end))
        else:
            indices.append(int(item) - 1)
    return np.asarray(sorted(set(i for i in indices if i >= 0)), dtype=int)


_CHANNEL_SORT = {
    "s": 0,
    "px": 10,
    "py": 11,
    "pz": 12,
    "d-2": 20,
    "d-1": 21,
    "d0": 22,
    "d+1": 23,
    "d+2": 24,
}


def _sorted_channels():
    if projection_data is None:
        return []
    channels = projection_data["channels"].astype(str).tolist()
    return sorted(channels, key=lambda name: (_CHANNEL_SORT.get(name, 100), name))

spin_indices = sorted(
    int(key.rsplit("_", 1)[-1])
    for key in data
    if key.startswith("weights_spin_")
)

print(f"SCF workchain PK: {workcalc.pk}")
print("label:", workcalc.label)
print("primitive vectors [Angstrom]:")
print(data["primitive_vectors"])
print("supercell matrix M:")
print(data["supercell_matrix"])
print("det(M):", int(round(abs(np.linalg.det(data["supercell_matrix"])))))
print("folded k-points:", len(data["k_frac_folded"]))
print("k-points on path:", len(data["path_k_indices"]))
print("path:", "-".join(data["path_labels"].astype(str)))
if "primitive_basis_atom_indices" in data:
    print("primitive basis atom indices:", " ".join(str(int(x)) for x in data["primitive_basis_atom_indices"]))
for spin in spin_indices:
    disp_key = f"atom_mapping_displacements_spin_{spin}"
    if disp_key in data:
        disp = np.linalg.norm(data[disp_key], axis=1)
        print(
            f"atom mapping spin {spin}: "
            f"max {disp.max():.6g} A, mean {disp.mean():.6g} A, "
            f"worst atom {int(np.argmax(disp)) + 1}"
        )
if projection_data is not None:
    print("atom/orbital projections:", len(projection_data["projection"]), "non-zero entries")
    print("projection channels:", ", ".join(projection_data["channels"].astype(str)))


In [ ]:
def _canonical_frac(values, tol=1.0e-8):
    folded = np.mod(np.asarray(values, dtype=float), 1.0)
    folded[np.isclose(folded, 1.0, atol=tol) | np.isclose(folded, 0.0, atol=tol)] = 0.0
    return folded


def _lattice_matrix(vectors):
    vec = np.asarray(vectors, dtype=float)
    dim = int(data["dim"])
    return vec[:dim, :dim].T


def _reciprocal_vectors(primitive_vectors):
    return 2.0 * np.pi * np.linalg.inv(_lattice_matrix(primitive_vectors)).T


def _kfrac_to_cart(k_frac, primitive_vectors):
    k_frac = np.asarray(k_frac, dtype=float)
    bmat = _reciprocal_vectors(primitive_vectors)
    k_dim = k_frac @ bmat.T
    cart = np.zeros((len(k_dim), 3))
    cart[:, : bmat.shape[0]] = k_dim
    return cart


def _clip_polygon_halfplane(poly, normal, offset, tol=1.0e-12):
    if len(poly) == 0:
        return poly
    clipped = []
    prev = poly[-1]
    prev_inside = np.dot(prev, normal) <= offset + tol
    for curr in poly:
        curr_inside = np.dot(curr, normal) <= offset + tol
        if curr_inside != prev_inside:
            denom = float(np.dot(curr - prev, normal))
            if abs(denom) > tol:
                t = float((offset - np.dot(prev, normal)) / denom)
                clipped.append(prev + t * (curr - prev))
        if curr_inside:
            clipped.append(curr)
        prev = curr
        prev_inside = curr_inside
    return np.asarray(clipped)


def _wigner_seitz_cell_2d(recip_vectors):
    b1, b2 = np.asarray(recip_vectors[:2, :2], dtype=float)
    neighbors = []
    for i in range(-2, 3):
        for j in range(-2, 3):
            if i == 0 and j == 0:
                continue
            g = i * b1 + j * b2
            neighbors.append(g)
    neighbors = sorted(neighbors, key=lambda g: np.dot(g, g))
    radius = 2.0 * max(np.linalg.norm(b1), np.linalg.norm(b2), 1.0)
    poly = np.asarray([[-radius, -radius], [radius, -radius], [radius, radius], [-radius, radius]], dtype=float)
    for g in neighbors:
        poly = _clip_polygon_halfplane(poly, g, 0.5 * float(np.dot(g, g)))
    center = poly.mean(axis=0)
    order = np.argsort(np.arctan2(poly[:, 1] - center[1], poly[:, 0] - center[0]))
    return poly[order]


def _draw_arrow(ax, vector, label, color):
    vector = np.asarray(vector, dtype=float)[:2]
    ax.arrow(0.0, 0.0, vector[0], vector[1], head_width=0.04 * max(np.linalg.norm(vector), 1.0),
             length_includes_head=True, color=color, linewidth=1.4)
    ax.text(vector[0], vector[1], label, color=color, ha="left", va="bottom")


def _pad_axes(ax, points):
    points = np.asarray(points, dtype=float)
    if points.size == 0:
        return
    xmin, ymin = points.min(axis=0)
    xmax, ymax = points.max(axis=0)
    pad = 0.08 * max(xmax - xmin, ymax - ymin, 1.0)
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)


def _plot_lattice_diagnostics(ax_real, ax_rec, primitive_vectors, supercell_vectors, hs_points, path_labels, k_frac, path_q):
    avec = np.asarray(primitive_vectors, dtype=float)[:2, :2]
    svec = np.asarray(supercell_vectors, dtype=float)[:2, :2]
    bvec = _reciprocal_vectors(primitive_vectors)[:2, :2].T

    real_cell = np.asarray([[0, 0], avec[0], avec[0] + avec[1], avec[1], [0, 0]], dtype=float)
    super_cell = np.asarray([[0, 0], svec[0], svec[0] + svec[1], svec[1], [0, 0]], dtype=float)
    ax_real.plot(real_cell[:, 0], real_cell[:, 1], color="black", linewidth=1.1, label="primitive cell")
    ax_real.plot(super_cell[:, 0], super_cell[:, 1], color="tab:gray", linewidth=1.0, linestyle="--", label="supercell")
    _draw_arrow(ax_real, avec[0], "a1", "tab:blue")
    _draw_arrow(ax_real, avec[1], "a2", "tab:orange")
    ax_real.set_title("Direct space")
    ax_real.set_xlabel("x [Angstrom]")
    ax_real.set_ylabel("y [Angstrom]")
    ax_real.set_aspect("equal", adjustable="box")
    ax_real.legend(loc="best", fontsize=8)
    _pad_axes(ax_real, np.vstack([real_cell, super_cell]))

    bz = _wigner_seitz_cell_2d(bvec)
    bz_closed = np.vstack([bz, bz[0]])
    ax_rec.plot(bz_closed[:, 0], bz_closed[:, 1], color="black", linewidth=1.2, label="1st BZ")
    _draw_arrow(ax_rec, bvec[0], "b1", "tab:blue")
    _draw_arrow(ax_rec, bvec[1], "b2", "tab:orange")
    k_cart = _kfrac_to_cart(k_frac, primitive_vectors)[:, :2]
    ax_rec.scatter(k_cart[:, 0], k_cart[:, 1], s=18, alpha=0.35, label="folded k-points")
    if len(path_q):
        path_cart = _kfrac_to_cart(path_q, primitive_vectors)[:, :2]
        ax_rec.scatter(path_cart[:, 0], path_cart[:, 1], s=42, color="tab:red", label="on path")
    nodes = []
    for label in path_labels:
        if label in hs_points:
            point = _kfrac_to_cart(np.asarray([hs_points[label]]), primitive_vectors)[0, :2]
            nodes.append(point)
            ax_rec.text(point[0], point[1], label, ha="center", va="bottom")
    if nodes:
        nodes = np.asarray(nodes)
        ax_rec.plot(nodes[:, 0], nodes[:, 1], color="black", linewidth=1.2, label="path")
    ax_rec.set_title("Reciprocal space")
    ax_rec.set_xlabel("kx [1/Angstrom]")
    ax_rec.set_ylabel("ky [1/Angstrom]")
    ax_rec.set_aspect("equal", adjustable="box")
    ax_rec.legend(loc="best", fontsize=8)
    node_points = np.asarray(nodes) if len(nodes) else bz
    _pad_axes(ax_rec, np.vstack([bz, bvec, k_cart, node_points]))


def _display_high_symmetry_points():
    dim = int(data["dim"])
    lattice_type = str(np.asarray(data.get("lattice_type", "")).item()).lower()
    if dim == 1:
        return {"G": np.array([0.0]), "X": np.array([0.5])}
    if lattice_type in {"hex", "hexagonal", "graphene"}:
        return {
            "G": np.array([0.0, 0.0]),
            "K": np.array([2.0 / 3.0, 1.0 / 3.0]),
            "M": np.array([0.5, 0.0]),
        }
    points = {}
    for label in data["path_labels"].astype(str):
        key = f"hs_point_{label}"
        if key in data:
            points[label] = np.asarray(data[key], dtype=float)
    return points


def _kpath_axis(points, path_labels, primitive_vectors):
    nodes_frac = [points[label] for label in path_labels]
    nodes_cart = _kfrac_to_cart(np.asarray(nodes_frac), primitive_vectors)
    x_nodes = [0.0]
    for i in range(1, len(nodes_cart)):
        x_nodes.append(x_nodes[-1] + float(np.linalg.norm(nodes_cart[i] - nodes_cart[i - 1])))
    return np.asarray(x_nodes), nodes_cart


def _project_kpoints_to_path(k_frac_points, points, path_labels, primitive_vectors, tol_cart=1.0e-6):
    k_frac_points = np.asarray(k_frac_points, dtype=float)
    dim = k_frac_points.shape[1]
    x_nodes, nodes_cart = _kpath_axis(points, path_labels, primitive_vectors)
    integer_shifts = [np.asarray(s, dtype=float) - 1.0 for s in np.ndindex(*([3] * dim))]
    projected = []
    for ik, q in enumerate(k_frac_points):
        best = None
        for shift in integer_shifts:
            q_equiv = q + shift
            q_cart = _kfrac_to_cart(np.asarray([q_equiv]), primitive_vectors)[0]
            for iseg in range(len(path_labels) - 1):
                a_cart = nodes_cart[iseg]
                b_cart = nodes_cart[iseg + 1]
                v = b_cart - a_cart
                denom = float(np.dot(v, v))
                if denom == 0.0:
                    continue
                t = float(np.dot(q_cart - a_cart, v) / denom)
                if t < -1.0e-10 or t > 1.0 + 1.0e-10:
                    continue
                closest = a_cart + t * v
                dist = float(np.linalg.norm(q_cart - closest))
                if dist <= tol_cart:
                    x = x_nodes[iseg] + t * float(np.linalg.norm(v))
                    candidate = (dist, ik, x, iseg, t, q_equiv)
                    if best is None or candidate[0] < best[0]:
                        best = candidate
        if best is not None:
            projected.append(best)
    if not projected:
        dim = k_frac_points.shape[1]
        return np.array([], dtype=int), np.array([]), np.empty((0, dim)), x_nodes
    return (
        np.asarray([p[1] for p in projected], dtype=int),
        np.asarray([p[2] for p in projected], dtype=float),
        np.asarray([p[5] for p in projected], dtype=float),
        x_nodes,
    )


path_labels = data["path_labels"].astype(str)
hs_points = _display_high_symmetry_points()
missing_points = [label for label in path_labels if label not in hs_points]
if missing_points:
    path_indices = data["path_k_indices"].astype(int)
    path_x = data["path_x"]
    path_q_equiv = data["path_q_equiv"]
    x_ticks = data["x_ticks"]
else:
    path_indices, path_x, path_q_equiv, x_ticks = _project_kpoints_to_path(
        data["k_frac_folded"], hs_points, path_labels, data["primitive_vectors"]
    )
x_labels = data["x_tick_labels"].astype(str)

all_energies = np.concatenate([
    data[f"evals_ev_spin_{spin}"] - float(data["ref_energy_ev"])
    for spin in spin_indices
])
finite_energies = all_energies[np.isfinite(all_energies)]
default_emin = float(np.floor(finite_energies.min())) if len(finite_energies) else -10.0
default_emax = float(np.ceil(finite_energies.max())) if len(finite_energies) else 10.0

spin_widget = ipw.Dropdown(options=spin_indices, value=spin_indices[0], description="Spin:")
spin_view = ipw.ToggleButtons(
    options=[("selected", "single"), ("side by side", "both")],
    value="both" if len(spin_indices) > 1 else "single",
    description="Spin view:",
    disabled=len(spin_indices) < 2,
)
marker_scale = ipw.FloatSlider(value=220.0, min=10.0, max=800.0, step=10.0, description="Scale:")
emin_widget = ipw.FloatText(value=default_emin, description="E min [eV]:", layout={"width": "180px"})
emax_widget = ipw.FloatText(value=default_emax, description="E max [eV]:", layout={"width": "180px"})
show_kmap = ipw.Checkbox(value=True, description="Show k-map")
out = ipw.Output()
projection_controls = []
projection_box = ipw.VBox([])
projection_message = ipw.HTML("")
add_projection_button = ipw.Button(description="Add projection", button_style="info")


def _projection_channels(control):
    return [name for name, checkbox in control["channels"].items() if checkbox.value]


def _refresh_projection_box():
    projection_box.children = [control["box"] for control in projection_controls]
    if projection_data is None:
        projection_message.value = "<i>No atom/orbital projection data were retrieved for this run.</i>"
    elif not projection_controls:
        projection_message.value = ""


def _remove_projection(control):
    if control in projection_controls:
        projection_controls.remove(control)
    _refresh_projection_box()
    _plot()


def _make_projection(default_color="#d62728"):
    channels = _sorted_channels()
    channel_checks = {
        name: ipw.Checkbox(value=True, description=name, indent=False, layout={"width": "70px"})
        for name in channels
    }
    control = {
        "enabled": ipw.Checkbox(value=True, description="show", indent=False),
        "atoms": ipw.Text(value="", placeholder="all atoms or 1..24", description="Atoms:", layout={"width": "220px"}),
        "color": ipw.ColorPicker(value=default_color, description="Color:", concise=False, layout={"width": "180px"}),
        "scale": ipw.FloatSlider(value=260.0, min=5.0, max=1200.0, step=5.0, description="Scale:", readout_format=".0f", layout={"width": "260px"}),
        "channels": channel_checks,
        "delete": ipw.Button(description="Delete", button_style="danger", layout={"width": "80px"}),
    }
    channel_box = ipw.HBox(list(channel_checks.values()), layout={"flex_flow": "row wrap"})
    control["box"] = ipw.VBox([
        ipw.HBox([control["enabled"], control["atoms"], control["color"], control["scale"], control["delete"]]),
        channel_box,
    ])
    for widget in [control["enabled"], control["atoms"], control["color"], control["scale"], *channel_checks.values()]:
        widget.observe(_plot, "value")
    control["delete"].on_click(lambda _: _remove_projection(control))
    return control


def _add_projection(_=None):
    if projection_data is None:
        projection_message.value = "<i>No atom/orbital projection data were retrieved for this run.</i>"
        return
    colors = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e"]
    projection_controls.append(_make_projection(colors[len(projection_controls) % len(colors)]))
    _refresh_projection_box()
    _plot()


def _draw_unfolding_axis(ax, spin, emin, emax, show_ylabel=True):
    weights = data[f"weights_spin_{spin}"]
    energies = data[f"evals_ev_spin_{spin}"] - float(data["ref_energy_ev"])
    energy_mask = (energies >= emin) & (energies <= emax)

    for ik, x in zip(path_indices, path_x):
        sizes = marker_scale.value * np.maximum(weights[ik], 0.0)
        mask = (sizes > 1.0e-8) & energy_mask
        ax.scatter(
            np.full(np.count_nonzero(mask), x),
            energies[mask],
            s=sizes[mask],
            alpha=0.55,
            color="black",
            edgecolors="none",
        )

    for control in projection_controls:
        if not control["enabled"].value:
            continue
        channels = _projection_channels(control)
        if not channels:
            continue
        try:
            atom_indices = _parse_atom_indices(control["atoms"].value)
        except Exception as exc:
            projection_message.value = f"<span style='color:red'>Invalid atom selection:</span> {exc}"
            continue
        projection = projection_weights_by_mo(spin, atom_indices=atom_indices, channels=channels)
        if projection is None:
            continue
        for ik, x in zip(path_indices, path_x):
            projected_sizes = control["scale"].value * np.maximum(weights[ik], 0.0) * np.maximum(projection, 0.0)
            mask = (projected_sizes > 1.0e-8) & energy_mask
            ax.scatter(
                np.full(np.count_nonzero(mask), x),
                energies[mask],
                s=projected_sizes[mask],
                alpha=0.75,
                color=control["color"].value,
                edgecolors="none",
            )

    for xt in x_ticks:
        ax.axvline(xt, linewidth=0.8, alpha=0.4)
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xticks(x_ticks, x_labels)
    ax.set_xlim(x_ticks[0], x_ticks[-1])
    ax.set_ylim(emin, emax)
    ax.set_xlabel("primitive-cell k-path")
    if show_ylabel:
        ax.set_ylabel("Energy - reference [eV]")
    ax.set_title(f"spin {spin}")


def _plot(_=None):
    with out:
        out.clear_output(wait=True)
        emin = float(emin_widget.value)
        emax = float(emax_widget.value)
        if emin > emax:
            emin, emax = emax, emin

        if projection_data is not None:
            projection_message.value = ""

        if show_kmap.value and int(data["dim"]) == 2:
            fig_diag, (ax_real, ax_rec) = plt.subplots(1, 2, figsize=(12.0, 4.6))
            _plot_lattice_diagnostics(
                ax_real,
                ax_rec,
                data["primitive_vectors"],
                data["supercell_vectors"],
                hs_points,
                path_labels,
                data["k_frac_folded"],
                path_q_equiv,
            )
            fig_diag.tight_layout()
            plt.show()

        fig, ax = plt.subplots(figsize=(7, 4))
        energy_mask = (energies >= emin) & (energies <= emax)
        for ik, x in zip(path_indices, path_x):
            sizes = marker_scale.value * np.maximum(weights[ik], 0.0)
            mask = (sizes > 1.0e-8) & energy_mask
            ax.scatter(np.full(np.count_nonzero(mask), x), energies[mask], s=sizes[mask], alpha=0.55, color="black")

        for control in projection_controls:
            if not control["enabled"].value:
                continue
            channels = _projection_channels(control)
            if not channels:
                continue
            try:
                atom_indices = _parse_atom_indices(control["atoms"].value)
            except Exception as exc:
                projection_message.value = f"<span style='color:red'>Invalid atom selection:</span> {exc}"
                continue
            projection = projection_weights_by_mo(spin, atom_indices=atom_indices, channels=channels)
            if projection is None:
                continue
            for ik, x in zip(path_indices, path_x):
                projected_sizes = control["scale"].value * np.maximum(weights[ik], 0.0) * np.maximum(projection, 0.0)
                mask = (projected_sizes > 1.0e-8) & energy_mask
                ax.scatter(
                    np.full(np.count_nonzero(mask), x),
                    energies[mask],
                    s=projected_sizes[mask],
                    alpha=0.75,
                    color=control["color"].value,
                    edgecolors="none",
                )
        for xt in x_ticks:
            ax.axvline(xt, linewidth=0.8, alpha=0.4)
        ax.axhline(0.0, linestyle="--", linewidth=1)
        ax.set_xticks(x_ticks, x_labels)
        ax.set_xlim(x_ticks[0], x_ticks[-1])
        ax.set_ylim(emin, emax)
        ax.set_xlabel("primitive-cell k-path")
        ax.set_ylabel("Energy - reference [eV]")
        ax.set_title(f"Unfolded weights, spin {spin}")
        fig.tight_layout()
        plt.show()

for widget in (spin_widget, marker_scale, emin_widget, emax_widget, show_kmap):
    widget.observe(_plot, "value")
add_projection_button.on_click(_add_projection)
controls = ipw.VBox([
    ipw.HBox([spin_widget, marker_scale, emin_widget, emax_widget, show_kmap]),
    ipw.HBox([add_projection_button, projection_message]),
    projection_box,
])
display(controls, out)
_refresh_projection_box()
_plot()
